# SNR-Auswertung der Trainingsdaten (ohne Datenveraenderung)

Dieses Notebook analysiert nur den Ist-Zustand deiner Daten (noisy/clean), damit du fair mit anderen Papers vergleichen kannst.
Es veraendert keine Audiodateien und fuehrt kein Training aus.

## 1) Umgebung und Pfade

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path('/workspaces/MetricGAN-KAN-LwF')
PROJECT = ROOT / 'metricgan-lwf'
hparams_yaml = PROJECT / 'hparams' / 'train.yaml'

# Nur setzen, wenn train.yaml data_folder als Placeholder enthaelt
data_folder_override = None

# Fuer schnellen Test optional begrenzen, sonst None
max_files_per_split = None

print('Python:', sys.version.split()[0])
print('Projekt:', PROJECT)
print('hparams:', hparams_yaml)
print('CUDA sichtbar:', os.environ.get('CUDA_VISIBLE_DEVICES', '(nicht gesetzt)'))

In [ ]:
import importlib

required = ['numpy', 'pandas', 'soundfile', 'matplotlib']
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print('Fehlende Pakete:', missing)
    print('Bitte installieren: pip install ' + ' '.join(missing))

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import display

pesq_available = importlib.util.find_spec('pesq') is not None
stoi_available = importlib.util.find_spec('pystoi') is not None

if pesq_available:
    from pesq import pesq
if stoi_available:
    from pystoi import stoi

print('Imports OK.')
print('PESQ verfuegbar:', pesq_available)
print('STOI verfuegbar:', stoi_available)

## 2) Abhaengigkeiten und Imports

## 3) Konfiguration aus train.yaml lesen

In [ ]:
import json
import re


def parse_key_from_yaml(text, key):
    pat = rf"^{key}:\s*(.+)$"
    for line in text.splitlines():
        m = re.match(pat, line.strip())
        if m:
            return m.group(1).strip()
    return None


def strip_sb_ref(value):
    if value is None:
        return None
    value = value.replace('!ref', '').strip()
    if value.startswith('<') and value.endswith('>'):
        value = value[1:-1]
    return value


def load_hparams_paths(yaml_path):
    txt = Path(yaml_path).read_text(encoding='utf-8')
    out = {
        'data_folder': strip_sb_ref(parse_key_from_yaml(txt, 'data_folder')),
        'train_annotation': strip_sb_ref(parse_key_from_yaml(txt, 'train_annotation')),
        'valid_annotation': strip_sb_ref(parse_key_from_yaml(txt, 'valid_annotation')),
        'test_annotation': strip_sb_ref(parse_key_from_yaml(txt, 'test_annotation')),
    }
    return out


cfg = load_hparams_paths(hparams_yaml)
if data_folder_override:
    cfg['data_folder'] = data_folder_override

print('Konfiguration:')
for k, v in cfg.items():
    print(f'- {k}: {v}')

In [ ]:
def replace_data_root(path_like, data_root):
    if path_like is None:
        return None
    data_root = '' if data_root is None else str(data_root)
    return str(Path(path_like.replace('{data_root}', data_root)))


def load_annotation_json(json_path):
    p = Path(json_path)
    if not p.exists():
        raise FileNotFoundError(f'Annotation nicht gefunden: {p}')
    with p.open('r', encoding='utf-8') as f:
        data = json.load(f)

    rows = []
    for utt_id, item in data.items():
        rows.append({
            'id': utt_id,
            'noisy_wav': item.get('noisy_wav'),
            'clean_wav': item.get('clean_wav'),
        })
    return pd.DataFrame(rows)


def read_audio_mono(path):
    wav, sr = sf.read(path)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    return wav.astype(np.float64), int(sr)


def snr_db(clean, noisy, eps=1e-10):
    noise = noisy - clean
    p_sig = float(np.mean(clean ** 2))
    p_noise = float(np.mean(noise ** 2))
    return 10.0 * np.log10((p_sig + eps) / (p_noise + eps))

## 4) Annotationen laden (nur Pfadinspektion, keine Datenaenderung)

## 5) Noisy-Baseline auswerten (SNR, optional PESQ/STOI)

In [ ]:
ann_paths = {
    'train': replace_data_root(cfg['train_annotation'], cfg['data_folder']),
    'valid': replace_data_root(cfg['valid_annotation'], cfg['data_folder']),
    'test': replace_data_root(cfg['test_annotation'], cfg['data_folder']),
}

splits = {}
for split, ann_path in ann_paths.items():
    try:
        df = load_annotation_json(ann_path)
        if max_files_per_split is not None:
            df = df.head(int(max_files_per_split)).copy()
        df['noisy_path'] = df['noisy_wav'].apply(lambda x: replace_data_root(x, cfg['data_folder']))
        df['clean_path'] = df['clean_wav'].apply(lambda x: replace_data_root(x, cfg['data_folder']))
        splits[split] = df
        print(f'{split}: {len(df)} Paare | Annotation: {ann_path}')
    except Exception as e:
        print(f'{split}: konnte nicht geladen werden ({e})')

for split, df in splits.items():
    display(df.head(3))

In [ ]:
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x


def eval_split(df, split_name):
    rows = []
    skipped_missing = 0
    skipped_sr = 0
    skipped_len = 0
    skipped_short = 0

    for _, r in tqdm(df.iterrows(), total=len(df), desc=f'{split_name} eval'):
        noisy_path = Path(r['noisy_path'])
        clean_path = Path(r['clean_path'])

        if not noisy_path.exists() or not clean_path.exists():
            skipped_missing += 1
            continue

        try:
            noisy, sr_n = read_audio_mono(noisy_path)
            clean, sr_c = read_audio_mono(clean_path)
        except Exception:
            skipped_missing += 1
            continue

        # Keine Datenanpassung: unterschiedliche SR oder Laengen werden uebersprungen.
        if sr_n != sr_c:
            skipped_sr += 1
            continue
        if len(noisy) != len(clean):
            skipped_len += 1
            continue
        if len(noisy) < 16:
            skipped_short += 1
            continue

        row = {
            'id': r['id'],
            'split': split_name,
            'sr': sr_n,
            'n_samples': len(noisy),
            'snr_db': snr_db(clean, noisy),
            'noisy_path': str(noisy_path),
            'clean_path': str(clean_path),
        }

        if pesq_available and sr_n == 16000:
            try:
                row['pesq_noisy'] = pesq(16000, clean, noisy, 'wb')
            except Exception:
                row['pesq_noisy'] = np.nan
        else:
            row['pesq_noisy'] = np.nan

        if stoi_available:
            try:
                row['stoi_noisy'] = stoi(clean, noisy, sr_n, extended=False)
            except Exception:
                row['stoi_noisy'] = np.nan
        else:
            row['stoi_noisy'] = np.nan

        rows.append(row)

    out = pd.DataFrame(rows)
    skip_info = {
        'split': split_name,
        'input_pairs': len(df),
        'used_pairs': len(out),
        'skipped_missing_or_read_error': skipped_missing,
        'skipped_sr_mismatch': skipped_sr,
        'skipped_length_mismatch': skipped_len,
        'skipped_too_short': skipped_short,
    }
    return out, skip_info


all_rows = []
skip_rows = []
for split, df in splits.items():
    out, info = eval_split(df, split)
    all_rows.append(out)
    skip_rows.append(info)

metrics_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
skip_df = pd.DataFrame(skip_rows)

display(skip_df)

if metrics_df.empty:
    print('Keine auswertbaren Paare gefunden.')
else:
    summary = metrics_df.groupby('split')[['snr_db', 'pesq_noisy', 'stoi_noisy']].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(3)
    display(summary)

    bins = [-np.inf, -5, 0, 5, 10, 15, 20, np.inf]
    labels = ['<-5', '-5..0', '0..5', '5..10', '10..15', '15..20', '>20']
    metrics_df['snr_bin_db'] = pd.cut(metrics_df['snr_db'], bins=bins, labels=labels)

    bin_table = (
        metrics_df.groupby(['split', 'snr_bin_db'])
        .agg(
            n=('id', 'count'),
            snr_mean=('snr_db', 'mean'),
            pesq_mean=('pesq_noisy', 'mean'),
            stoi_mean=('stoi_noisy', 'mean'),
        )
        .reset_index()
    )
    display(bin_table)

    for split in sorted(metrics_df['split'].unique()):
        sub = metrics_df[metrics_df['split'] == split]
        print(f'\n{split}: niedrigste SNR-Beispiele')
        display(sub.nsmallest(10, 'snr_db')[['id', 'snr_db', 'pesq_noisy', 'stoi_noisy', 'noisy_path']])

In [ ]:
if metrics_df.empty:
    print('Keine Metriken vorhanden.')
else:
    splits_sorted = sorted(metrics_df['split'].unique())
    fig, axes = plt.subplots(len(splits_sorted), 1, figsize=(10, 3 * len(splits_sorted)), squeeze=False)
    for i, split in enumerate(splits_sorted):
        ax = axes[i, 0]
        sub = metrics_df[metrics_df['split'] == split]
        ax.hist(sub['snr_db'].dropna().values, bins=40, alpha=0.8, color='#1f77b4')
        ax.set_title(f'SNR-Verteilung ({split})')
        ax.set_xlabel('SNR [dB]')
        ax.set_ylabel('Anzahl')
    plt.tight_layout()
    plt.show()

## 6) Table-1-kompatibler Vergleich: Noisy vs MetricGAN vs MetricGAN-LwF

In [ ]:
# Ordner mit Enhanced-WAVs angeben.
# Erwartet Dateinamenformat: <id>.wav
metricgan_enhanced_dir = None      # z.B. '/workspaces/.../metricgan/results/.../enhanced_wavs'
metricgan_lwf_enhanced_dir = None  # z.B. '/workspaces/.../metricgan-lwf/results/.../enhanced_wavs'

# Vergleich auf diesen nominalen SNR-Stufen wie im Paper
target_snr_levels = [-12, -6, 0, 6, 12]
snr_tolerance_db = 1.0

# Normalerweise test verwenden fuer Papervergleich
eval_split_for_table = 'test'

# Mindestanzahl pro SNR-Stufe; darunter Warnung
min_samples_per_level = 20

print('Konfiguration gesetzt.')
print('Split:', eval_split_for_table)
print('SNR-Ziele:', target_snr_levels)
print('Toleranz ± dB:', snr_tolerance_db)

In [ ]:
def eval_enhanced_for_ids(base_df, enhanced_dir, model_name):
    if enhanced_dir is None:
        print(f'{model_name}: enhanced_dir ist None, uebersprungen.')
        return pd.DataFrame(columns=['id', 'split', 'pesq', 'stoi'])

    enhanced_dir = Path(enhanced_dir)
    rows = []

    for _, r in base_df.iterrows():
        clean_path = Path(r['clean_path'])
        enh_path = enhanced_dir / f"{r['id']}.wav"

        if (not clean_path.exists()) or (not enh_path.exists()):
            continue

        try:
            enh, sr_e = read_audio_mono(enh_path)
            clean, sr_c = read_audio_mono(clean_path)
        except Exception:
            continue

        if sr_e != sr_c:
            continue
        if len(enh) != len(clean):
            continue
        if len(enh) < 16:
            continue

        row = {
            'id': r['id'],
            'split': r['split'],
            'pesq': np.nan,
            'stoi': np.nan,
        }

        if pesq_available and sr_e == 16000:
            try:
                row['pesq'] = pesq(16000, clean, enh, 'wb')
            except Exception:
                row['pesq'] = np.nan

        if stoi_available:
            try:
                row['stoi'] = stoi(clean, enh, sr_e, extended=False)
            except Exception:
                row['stoi'] = np.nan

        rows.append(row)

    out = pd.DataFrame(rows)
    print(f'{model_name}: {len(out)} gueltige Enhanced-Paare ausgewertet.')
    return out


def summarize_levels(compare_df, level_col='snr_level'):
    agg = compare_df.groupby(level_col).agg(
        n=('id', 'count'),
        noisy_pesq=('pesq_noisy', 'mean'),
        noisy_stoi=('stoi_noisy', 'mean'),
        metricgan_pesq=('pesq_metricgan', 'mean'),
        metricgan_stoi=('stoi_metricgan', 'mean'),
        lwf_pesq=('pesq_lwf', 'mean'),
        lwf_stoi=('stoi_lwf', 'mean'),
    ).reset_index()
    return agg


if metrics_df.empty:
    print('Keine Baseline-Metriken vorhanden. Fuehre zuerst die Baseline-Zellen aus.')
else:
    base_split = metrics_df[metrics_df['split'] == eval_split_for_table].copy()

    if base_split.empty:
        print('Gewaehlter Split hat keine Daten:', eval_split_for_table)
    else:
        level_frames = []
        for lvl in target_snr_levels:
            sub = base_split[(base_split['snr_db'] >= (lvl - snr_tolerance_db)) & (base_split['snr_db'] <= (lvl + snr_tolerance_db))].copy()
            sub['snr_level'] = lvl
            level_frames.append(sub)

        level_df = pd.concat(level_frames, ignore_index=True) if level_frames else pd.DataFrame()

        if level_df.empty:
            print('Keine Samples in den SNR-Fenstern gefunden. Erhoehe ggf. snr_tolerance_db.')
        else:
            counts = level_df.groupby('snr_level')['id'].count().reset_index(name='n')
            display(counts)

            low_counts = counts[counts['n'] < min_samples_per_level]
            if not low_counts.empty:
                print('Warnung: Kleine Bin-Groessen gefunden:')
                display(low_counts)

            mg_df = eval_enhanced_for_ids(level_df[['id', 'split', 'clean_path']], metricgan_enhanced_dir, 'MetricGAN')
            lwf_df = eval_enhanced_for_ids(level_df[['id', 'split', 'clean_path']], metricgan_lwf_enhanced_dir, 'MetricGAN-LwF')

            compare = level_df[['id', 'split', 'snr_db', 'snr_level', 'pesq_noisy', 'stoi_noisy']].copy()
            compare = compare.merge(mg_df.rename(columns={'pesq': 'pesq_metricgan', 'stoi': 'stoi_metricgan'}), on=['id', 'split'], how='left')
            compare = compare.merge(lwf_df.rename(columns={'pesq': 'pesq_lwf', 'stoi': 'stoi_lwf'}), on=['id', 'split'], how='left')

            table1_like = summarize_levels(compare)
            table1_like = table1_like.sort_values('snr_level').reset_index(drop=True)

            avg_row = {
                'snr_level': 'Avg.',
                'n': int(table1_like['n'].sum()),
                'noisy_pesq': table1_like['noisy_pesq'].mean(),
                'noisy_stoi': table1_like['noisy_stoi'].mean(),
                'metricgan_pesq': table1_like['metricgan_pesq'].mean(),
                'metricgan_stoi': table1_like['metricgan_stoi'].mean(),
                'lwf_pesq': table1_like['lwf_pesq'].mean(),
                'lwf_stoi': table1_like['lwf_stoi'].mean(),
            }
            table1_like = pd.concat([table1_like, pd.DataFrame([avg_row])], ignore_index=True)

            display(table1_like.round(3))

            print('Hinweis: Vergleiche Modelle nur auf exakt dieser gemeinsamen ID-Menge pro SNR-Stufe.')